# Градиентный бустинг: точный прогноз по табличным данным

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Градиентный бустинг».

## Что делает метод

Градиентный бустинг предсказывает ответ по таблице измерений и сегодня даёт наиболее точные результаты на табличных данных в большинстве прикладных задач. В отличие от случайного леса, деревья строятся не независимо, а последовательно: каждое следующее дерево сосредоточено на тех наблюдениях, где предыдущие ошиблись сильнее всего.

Метод применим, когда есть таблица: строки — наблюдения, столбцы — измеренные признаки и непрерывная целевая величина или метка класса. Типичные научные задачи: предсказание свойства материала по составу, оценка отклика по условиям эксперимента, прогноз показателя по набору факторов.

В этом блокноте мы решим задачу регрессии: предскажем непрерывную величину и оценим качество прогноза. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте студента, который готовится к экзамену. После каждой тренировочной попытки он не повторяет всё подряд, а сосредотачивается только на тех задачах, которые решил неправильно. Следующая попытка — снова разбор своих ошибок. К концу подготовки он силён именно там, где сначала был слаб.

Градиентный бустинг работает по той же логике: деревья строятся последовательно, и каждое новое дерево целенаправленно исправляет ошибки предыдущего ансамбля. Поэтому бустинг часто точнее случайного леса — но и более чувствителен к настройке.

**Ключевые понятия, которые встретятся в блокноте:**

- **Признак** — один измеренный параметр наблюдения (столбец таблицы).
- **Обучающая выборка** — часть данных, на которой модель учится. **Тестовая выборка** — отложенная часть для оценки качества.
- **Ансамбль** — набор нескольких моделей, чьи ответы объединяются для получения итогового предсказания.
- **Переобучение** — ситуация, когда модель запомнила шум обучающих данных и плохо работает на новых.
- **Гиперпараметр** — настройка алгоритма до обучения: скорость обучения, число деревьев, глубина.
- **Ранняя остановка** — автоматическое прекращение обучения, когда качество на проверочной выборке перестаёт улучшаться; защищает от переобучения.
- **Метрика** — числовая мера качества предсказания (здесь: MAE и R²).
- **Регрессия** — задача предсказания непрерывной числовой величины (в отличие от классификации, где ответ — метка класса).

## 1. Установка библиотек

Используем встроенный в `scikit-learn` градиентный бустинг (`HistGradientBoostingRegressor`) — он не требует сторонних пакетов и работает быстро.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`; вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем открытый набор о жилье в Калифорнии (`california_housing` из `scikit-learn`): около 20 тысяч наблюдений и 8 числовых признаков (доход, возраст домов, координаты и т. д.). Целевая величина — медианная стоимость жилья в районе.

Это задача **регрессии**: нужно предсказать непрерывное число, а не метку класса. Ячейка ниже загружает данные и показывает первые строки таблицы.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
X = data.data           # таблица признаков
y = data.target         # целевая непрерывная величина
feature_names = list(X.columns)

print(f"Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}")
X.head()

## 4. Применение метода

**Шаг 4.1. Разбивка на обучающую и тестовую выборки.**

Откладываем 25 % данных в тестовую выборку — модель увидит их только один раз при финальной оценке.

**Шаг 4.2. Обучение градиентного бустинга с ранней остановкой.**

Используем `HistGradientBoostingRegressor` — быстрая реализация бустинга на гистограммах из scikit-learn. Ключевые параметры:
- `learning_rate=0.1` — **скорость обучения** (гиперпараметр): насколько сильно каждое новое дерево корректирует ансамбль. Меньше — устойчивее, но нужно больше деревьев.
- `max_iter=500` — максимальное число деревьев (итераций).
- `early_stopping=True` — **ранняя остановка**: обучение прекратится, когда качество на внутренней проверочной выборке не улучшается `n_iter_no_change=20` итераций подряд. Это предотвращает переобучение автоматически.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

# learning_rate - скорость обучения; ранняя остановка защищает от переобучения.
model = HistGradientBoostingRegressor(
    learning_rate=0.1, max_iter=500,
    early_stopping=True, validation_fraction=0.15,
    n_iter_no_change=20, random_state=42)
model.fit(X_train, y_train)
print(f"Фактически построено деревьев: {model.n_iter_}")

**Шаг 4.3. Оценка качества на тестовой выборке.**

Ячейка ниже печатает две метрики регрессии:
- **MAE (средняя абсолютная ошибка)** — насколько в среднем прогноз отклоняется от факта, в единицах целевой величины (стоимости жилья, $100 тыс.).
- **R² (коэффициент детерминации)** — доля разброса целевой величины, объяснённая моделью. Значение 0.8 означает, что модель объясняет 80 % вариации ответа. Максимум — 1.0.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)
print(f"Средняя абсолютная ошибка (MAE): {mean_absolute_error(y_test, y_pred):.3f}")
print(f"Коэффициент детерминации (R^2): {r2_score(y_test, y_pred):.3f}")

### Шаг 4.4. Качество прогноза и важность признаков

Ячейка ниже строит два графика:

1. **Прогноз против факта** — каждая точка соответствует одному наблюдению тестовой выборки. Ось X — истинное значение, ось Y — предсказанное. Идеальная модель даёт все точки на диагонали.
2. **Важность признаков** методом перестановок — насколько падает R² при случайном перемешивании каждого признака.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))

# Предсказание против факта
axes[0].scatter(y_test, y_pred, s=10, alpha=0.3, color=VIZ["series"][0])
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, color=VIZ["series"][2], linestyle="--",
             label="Идеальное совпадение")
axes[0].set_title("Прогноз против факта")
axes[0].set_xlabel("Истинное значение")
axes[0].set_ylabel("Предсказанное значение")
axes[0].legend(loc="upper left")

# Важность признаков (перестановочная)
perm = permutation_importance(model, X_test, y_test, n_repeats=10,
                              random_state=42)
order = np.argsort(perm.importances_mean)
axes[1].barh(np.array(feature_names)[order], perm.importances_mean[order],
             xerr=perm.importances_std[order], color=VIZ["series"][1])
axes[1].set_title("Важность признаков (перестановочная)")
axes[1].set_xlabel("Среднее падение качества")
axes[1].grid(True, axis="x")

fig.tight_layout()
plt.show()

**Как читать эти графики:**

- **Прогноз против факта (левый график)**: точки должны лежать вдоль пунктирной диагонали «идеальное совпадение». Вертикальный разброс вокруг диагонали — типичная ошибка модели. Если точки систематически выше или ниже диагонали в каком-то диапазоне — там модель смещена (например, недооценивает высокие значения).

- **Важность признаков (правый график)**: признаки упорядочены по значимости. Наверху — те, которые бустинг использует сильнее всего. Усы (линии погрешностей) показывают разброс оценки по повторным перемешиваниям: широкие усы — нестабильная оценка важности.

## 5. Интерпретация результата

- **MAE** — средняя абсолютная ошибка в единицах целевой величины: насколько в среднем прогноз отклоняется от факта.
- **R²** — доля разброса целевой величины, объяснённая моделью; значение ближе к единице означает более точный прогноз.
- **График «прогноз против факта»**: разброс точек вокруг диагонали показывает типичную ошибку; систематическое отклонение от диагонали указывает на смещение прогноза в части диапазона.
- **Важность признаков**: какие факторы сильнее всего влияют на предсказание.

Помните: качество бустинга зависит от подбора гиперпараметров, а оценки важности признаков не доказывают причинно-следственных связей. Метод плохо экстраполирует за пределы диапазона обучающих данных.

## 5б. Наглядный эксперимент: как бустинг последовательно исправляет ошибки

Построим синтетическую задачу регрессии с нелинейной зависимостью. На графике увидим, как прогноз меняется по мере добавления деревьев: 1 дерево — грубое приближение, 10 деревьев — лучше, 100 — почти точно. Это наглядно показывает идею «последовательного исправления ошибок».

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor

rng = np.random.default_rng(0)
X_s = np.sort(rng.uniform(0, 6, 200))
y_s = np.sin(X_s) + 0.5 * np.cos(2 * X_s) + rng.normal(0, 0.25, 200)

X_grid = np.linspace(0, 6, 400)
palette = get_palette(4)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X_s, y_s, s=15, alpha=0.4, color=palette[3], label="Данные")

for n, color, lw in [(1, palette[2], 1.5), (10, palette[1], 2.0), (100, palette[0], 2.5)]:
    gb = GradientBoostingRegressor(n_estimators=n, learning_rate=0.3,
                                   max_depth=3, random_state=42)
    gb.fit(X_s.reshape(-1, 1), y_s)
    ax.plot(X_grid, gb.predict(X_grid.reshape(-1, 1)),
            color=color, linewidth=lw, label=f"{n} {'дерево' if n==1 else 'деревьев'}")

ax.set_title("Как бустинг улучшает прогноз с ростом числа деревьев")
ax.set_xlabel("Признак")
ax.set_ylabel("Целевая величина")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

**Как читать этот график:**

Точки — синтетические данные с нелинейной зависимостью и шумом. Три линии — прогноз ансамбля при 1, 10 и 100 деревьях.

- 1 дерево: кривая грубая, ступенчатая — одно дерево видит только самые крупные тенденции.
- 10 деревьев: форма кривой угадывается, но ещё неточна.
- 100 деревьев: кривая хорошо следует за истинной зависимостью, не «запоминая» шум.

Это и есть суть бустинга: каждое дерево добавляет небольшую поправку, и совместно они строят гибкий прогноз.

## 6. Попробуйте на своих данных

Замените демонстрационный набор своими данными. Подготовьте таблицу, где строки — наблюдения, столбцы — признаки, и отдельный столбец — целевая непрерывная величина. `HistGradientBoostingRegressor` допускает пропуски в признаках, заполнять их заранее не обязательно.

1. Загрузите файл в Colab: панель слева, вкладка «Файлы», кнопка загрузки.
2. Снимите комментарии в ячейке ниже и укажите имя файла и название целевого столбца.
3. Последовательно выполните ячейки разделов 4 и 5. Для задачи классификации замените `HistGradientBoostingRegressor` на `HistGradientBoostingClassifier`.

## Поэкспериментируйте сами

Вернитесь к ячейке обучения (раздел 4) и попробуйте изменить параметры:

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `learning_rate` | Попробуйте 0.01 и 0.5 | Как меняется число построенных деревьев (ранняя остановка)? |
| `n_iter_no_change` | Измените с 20 на 5 | Останавливается ли обучение раньше? |
| `max_iter` | Уберите раннюю остановку (`early_stopping=False`) и задайте 50 | Переобучается ли модель при малом числе деревьев? |

На синтетическом примере (раздел 5б) попробуйте `learning_rate=1.0` при 100 деревьях — увидите переобучение: кривая начнёт «вилять» и воспроизводить шум данных.

## Краткие выводы

- Градиентный бустинг строит деревья последовательно: каждое следующее целенаправленно исправляет ошибки предыдущего ансамбля. Это обеспечивает высокую точность, но требует аккуратной настройки.
- Ранняя остановка — стандартный приём: она автоматически выбирает оптимальное число деревьев и предотвращает переобучение.
- Метод чувствителен к скорости обучения (`learning_rate`): маленькое значение требует больше деревьев, большое — риск переобучения.
- На большинстве табличных задач бустинг точнее случайного леса, но медленнее в обучении и сложнее в настройке.
- Важность признаков через перестановки надёжнее встроенных оценок; для глубокой интерпретации используйте SHAP-значения.
- Метод не экстраполирует: предсказания за пределами диапазона обучающих данных ненадёжны.

In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv("my_data.csv")          # ваш файл
# target_column = "целевая_величина"       # имя целевого столбца
#
# y = df[target_column]
# X = df.drop(columns=[target_column])
# feature_names = list(X.columns)
#
# print(f"Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}")
# Далее повторите ячейки раздела 4.